In [1]:
import re
import ipywidgets as widgets  # pip install ipywidgets
import datetime
widgets.IntSlider()
from IPython.display import display, clear_output
from bank_validator import BelgianBankAccountValidator as bbcv
from bank_account_balance import CompteBancaire
from creditcard_validator import validate_card as vcc

# État global pour conserver le compte en mémoire et l'historique de mes transactions
session = {
    "compte": None,
    "valide": False,
    "historique": []  # Historique des opérations
}

# Widgets
input_account = widgets.Text(description="Compte :", placeholder="BE....")
input_name = widgets.Text(description="Titulaire :", placeholder="Nom complet")
input_depot = widgets.IntText(description="Dépôt (€):", value=0)
input_retrait = widgets.IntText(description="Retrait (€):", value=0)
btn_valider = widgets.Button(description="Exécuter la transaction", button_style="success")
output = widgets.Output()

# Fonction pour valider et gérer les opérations
def valider_compte(b):
    with output:
        clear_output()
        num = re.sub(r'\s+', '', input_account.value)
        titulaire = input_name.value.strip()

        if not titulaire:
            print("❌ Le nom du titulaire est requis.")
            return

        # Initialisation du compte si nécessaire
        if session["compte"] is None or session["compte"].account_number != num:
            est_valide = bbcv.is_valid(num)
            session["valide"] = est_valide
            if not est_valide:
                print("❌ Numéro de compte invalide :", num)
                session["compte"] = None
                return
            session["compte"] = CompteBancaire(account_number=num, titulaire=titulaire)
            print("✅ Compte initialisé :", num)
            print("Titulaire :", titulaire)
        else:
            print("✅ Compte déjà initialisé :", num)
            print("Titulaire :", session['compte'].titulaire)

        compte = session["compte"]

        # Affichage solde initial
        compte.afficher_solde()

        # Dépôt(Il n'existe pas de montant minimum pour le dépôt)
        montant_depot = input_depot.value
        if montant_depot > 0:
            date_transaction = datetime.datetime.now()
            compte.deposer(montant_depot)
            session["historique"].append(
                f"Dépôt: {montant_depot:.2f}€ en date {date_transaction.strftime('%d-%m-%Y %H:%M:%S')}")
        else:
            print("💡 Aucun dépôt effectué.")

        # Retrait (On ne peut pas rétirer plus que l'on a sur son compte)
        montant_retrait = input_retrait.value
        if montant_retrait > 0:
            date_transaction = datetime.datetime.now()
            if montant_retrait > compte.solde:
                print(f"❌ Solde insuffisant pour retirer {montant_retrait:.2f}€ (solde actuel : {compte.solde:.2f}€), veuillez essayer un montant inférieur.")
                session["historique"].append(
                    f"❌ Retrait échoué: {montant_retrait:.2f}€ souhaité, solde disponible: {compte.solde:.2f}€ en date {date_transaction.strftime('%d-%m-%Y %H:%M:%S')}")
            else:
                compte.retirer(montant_retrait)
                session["historique"].append(
                    f"Retrait: {montant_retrait:.2f}€ en date {date_transaction.strftime('%d-%m-%Y %H:%M:%S')}")
        else:
            print("💡 Aucun retrait effectué.")

        # Affichage solde final
        compte.afficher_solde()

        # Affichage de l'historique des opérations
        print("\nHistorique des opérations :")
        for operation in session["historique"]:
            print(operation)

# Liaison du bouton
btn_valider.on_click(valider_compte)

# Affichage
display(input_account, input_name, input_depot, input_retrait, btn_valider, output)

# Bouton de réinitialisation
btn_reset = widgets.Button(description="Réinitialiser", button_style="danger")

# Fonction de réinitialisation
def reinitialiser_formulaire(b):
    input_account.value = ""
    input_name.value = ""
    input_depot.value = 0
    input_retrait.value = 0
    session["compte"] = None
    session["valide"] = False
    session["historique"].clear()
    with output:
        clear_output()
        print("🔄 Formulaire réinitialisé.")

# Liaison du bouton
btn_reset.on_click(reinitialiser_formulaire)



Valid Visa card number.
Valid MasterCard card number.
Valid American Express card number.
Valid Discover card number.
Invalid card number.


Text(value='', description='Compte :', placeholder='BE....')

Text(value='', description='Titulaire :', placeholder='Nom complet')

IntText(value=0, description='Dépôt (€):')

IntText(value=0, description='Retrait (€):')

Button(button_style='success', description='Exécuter la transaction', style=ButtonStyle())

Output()